# 04 â€” Groundsource: No-Flood Event Addition

**Purpose:** Generate negative training samples (no-flood rainfall events) to complement the
positive flood samples. Each negative sample is a real heavy-rain episode that occurred at
a flood-event location but did NOT coincide with a recorded flood.

**Sampling strategy:** A stratified sample of representative flood polygons is drawn from
events in 2018â€“2025. Stratification uses 18 buckets defined by:
- **Area**     : Small / Medium / Large (tertiles)
- **Urban %**  : Low (0â€“20%), Urban (20â€“60%), Highly Urban (60â€“100%)
- **Mechanism**: Pluvial (PFDI < 1) vs Fluvial/Extreme (PFDI â‰¥ 1)

For each sampled polygon, IMERG V07 data is queried over 2018â€“2025 to find months with
peak rainfall intensity above a minimum threshold, excluding a Â±3-day buffer around the
original flood event to avoid contamination.

**Input:**
- `outputs/groundsource_rain_master.parquet` â€” enriched flood events (from 03b)

**Output:**
- `outputs/no_flood_batches/no_flood_batch_*.parquet` â€” no-flood records per polygon batch
  Each record contains:
  - All intensity columns
  - `is_flood = 0`
  - `event_id` suffixed with `_noflood_YYYYMMDD` for traceability

In [ ]:
import os
import glob
import time
import uuid
import pickle
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import ee
from shapely import wkb as shapely_wkb

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH   = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_rain_master.parquet"
BATCH_DIR    = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\no_flood_batches"

GEE_PROJECT  = 'your-gee-project'     # Replace with your GEE project ID

# Stratified sampling parameters
MODERN_YEAR      = 2018    # only sample from 2018+ for better satellite coverage
SEARCH_YEARS     = list(range(2018, 2026))   # 2018â€“2025 inclusive
MAX_PER_BUCKET   = 8       # maximum polygons to sample per stratification bucket

# No-flood detection thresholds
THRESH_1H        = 7.0     # mm/hr â€” minimum 1-hour intensity to qualify as a storm
THRESH_24H       = 1.0     # mm/hr â€” minimum 24-hour intensity
BUFFER_DAYS      = 3       # days to exclude around a known flood event

# GEE parameters (same as 03a for consistency)
SCALE_DEG        = 0.1
HOURS_BEFORE     = 72
HOURS_AFTER      = 24
BATCH_SIZE_POLY  = 10      # polygons per GEE session batch
TIMEOUT_SCAN     = 120     # seconds â€” timeout for the GEE region-scan call
TIMEOUT_MATRIX   = 180     # seconds â€” timeout for the 3-D matrix download
WGS84_CRS        = "EPSG:4326"
DURATIONS_MIN    = [30, 60, 120, 240, 360, 720, 1440]

## 1. Initialise GEE and load data

In [ ]:
ee.Initialize(project=GEE_PROJECT)
print("Google Earth Engine initialised.")
os.makedirs(BATCH_DIR, exist_ok=True)

df = pd.read_parquet(INPUT_PATH)
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date']   = pd.to_datetime(df['end_date'])
print(f"Loaded {len(df):,} records")

## 2. Stratified sampling of representative polygons

In [ ]:
# Filter to modern events (better satellite coverage)
modern = df[df['start_date'].dt.year >= MODERN_YEAR].copy()
print(f"Events from {MODERN_YEAR}+: {len(modern):,}")

# --- 3-D stratification ---

# 1. Area tertiles
modern['area_bin'] = pd.qcut(
    modern['area_km2_mollweide'], q=3, labels=['Small', 'Medium', 'Large']
)

# 2. Urban percentage bins (matching the 0â€“20 / 20â€“60 / 60â€“100 scheme)
modern['urban_bin'] = pd.cut(
    modern['urban_percentage'],
    bins=[0, 20, 60, 100],
    labels=['Low-Urban', 'Urban', 'Highly-Urban'],
    include_lowest=True
)

# 3. Mechanism: pluvial (PFDI < 1) vs fluvial/extreme (PFDI >= 1)
modern['mechanism_bin'] = np.where(modern['PFDI_max'] < 1, 'Pluvial', 'Fluvial')

# Sample up to MAX_PER_BUCKET polygons from each of the 18 buckets
targets = (
    modern
    .groupby(['area_bin', 'urban_bin', 'mechanism_bin'], observed=True)
    .apply(lambda grp: grp.sample(n=min(MAX_PER_BUCKET, len(grp)), random_state=42))
    .reset_index(drop=True)
)

print(f"\nStratified sample: {len(targets)} polygons across "
      f"{targets.groupby(['area_bin','urban_bin','mechanism_bin'], observed=True).ngroups} buckets")
print(targets.groupby(['area_bin', 'urban_bin', 'mechanism_bin'], observed=True).size())

## 3. Helper functions

In [ ]:
def peak_rolling_mean(series, n_steps):
    """Return the maximum of a rolling mean of width n_steps over a 1-D array."""
    if series is None or len(series) < n_steps:
        return np.nan
    kernel = np.ones(n_steps) / n_steps
    rolled = np.convolve(series, kernel, mode='valid')
    return float(rolled.max()) if len(rolled) > 0 else np.nan


def rasterize_polygon(wkb_bytes, origin_lon, origin_lat, n_lon, n_lat):
    """Create a binary spatial mask (n_lat Ã— n_lon) for a polygon."""
    from rasterio.transform import from_origin
    from rasterio.features import rasterize
    geom      = shapely_wkb.loads(wkb_bytes)
    transform = from_origin(origin_lon, origin_lat + n_lat * SCALE_DEG, SCALE_DEG, SCALE_DEG)
    return rasterize(
        [(geom.__geo_interface__, 1)],
        out_shape=(n_lat, n_lon),
        transform=transform,
        fill=0, dtype='uint8', all_touched=True
    )


def extract_rain_matrix(wkb_bytes, storm_date, product='final'):
    """
    Download an IMERG rainfall matrix centred on a detected storm date.

    Returns (matrix, mask, meta) or (None, None, None) on failure.
    The interface is identical to 03a so intensity computation can be reused.
    """
    collection_id = (
        'NASA/GPM_L3/IMERG_V07/FINAL/HALF_HOURLY' if product == 'final'
        else 'NASA/GPM_L3/IMERG_V07/LATE/HALF_HOURLY'
    )
    t_start = storm_date - timedelta(hours=HOURS_BEFORE)
    t_end   = storm_date + timedelta(hours=HOURS_AFTER)

    geom = shapely_wkb.loads(wkb_bytes)
    minx, miny, maxx, maxy = geom.bounds
    buf    = SCALE_DEG
    region = ee.Geometry.BBox(minx - buf, miny - buf, maxx + buf, maxy + buf)

    ic = (
        ee.ImageCollection(collection_id)
        .filterDate(t_start.strftime('%Y-%m-%dT%H:%M:%S'),
                    t_end.strftime('%Y-%m-%dT%H:%M:%S'))
        .filterBounds(region)
        .select('precipitationCal')
    )

    n_images = ic.size().getInfo()
    if n_images == 0:
        return None, None, None

    image_list = ic.toList(n_images)
    slices, timestamps = [], []
    n_lon = n_lat = origin_lon = origin_lat = None

    for i in range(n_images):
        img  = ee.Image(image_list.get(i))
        info = img.sampleRectangle(region=region, defaultValue=0).getInfo()
        arr  = np.array(info['properties']['precipitationCal'], dtype=np.float32)
        if n_lat is None:
            n_lat, n_lon = arr.shape
            coords = info.get('geometry', {}).get('coordinates', [[]])[0]
            if coords:
                lons = [c[0] for c in coords]
                lats = [c[1] for c in coords]
                origin_lon, origin_lat = min(lons), max(lats)
            else:
                origin_lon, origin_lat = minx - buf, maxy + buf
        ts = img.get('system:time_start').getInfo()
        timestamps.append(datetime.utcfromtimestamp(ts / 1000).strftime('%Y-%m-%dT%H:%M:%S'))
        slices.append(arr)

    matrix = np.stack(slices, axis=0)
    mask   = rasterize_polygon(wkb_bytes, origin_lon, origin_lat, n_lon, n_lat)
    meta   = {'origin_lon': origin_lon, 'origin_lat': origin_lat,
              'scale_deg': SCALE_DEG, 'shape': matrix.shape, 'timestamps': timestamps}
    return matrix, mask, meta


def compute_intensities(matrix, mask):
    """
    Compute spatial-mean peak intensities for all 7 durations from a single event's arrays.

    Returns a dict mapping duration column names to float values.
    """
    n_pixels = mask.sum()
    if n_pixels == 0:
        return {f"{d}_max_rainfall_intens": np.nan for d in DURATIONS_MIN}

    series = (matrix * mask[np.newaxis, :, :]).sum(axis=(1, 2)) / n_pixels
    result = {}
    for dur_min in DURATIONS_MIN:
        n_steps = dur_min // 30
        kernel  = np.ones(n_steps) / n_steps
        rolled  = np.convolve(series, kernel, mode='valid')
        result[f"{dur_min}_max_rainfall_intens"] = float(rolled.max()) if len(rolled) > 0 else np.nan
    return result


def scan_for_storms(wkb_bytes, flood_dates_set):
    """
    Scan all months in SEARCH_YEARS for heavy-rain episodes at a given polygon location.

    Steps:
    1. For each year-month, find the date with maximum mean precipitation.
    2. Keep only dates where the 1-hr peak intensity > THRESH_1H.
    3. Exclude dates within BUFFER_DAYS of any known flood event at this location.

    Args:
        wkb_bytes       : bytes â€” WKB polygon geometry
        flood_dates_set : set of datetime.date â€” known flood dates at this polygon

    Returns:
        list of datetime objects â€” candidate storm dates
    """
    from datetime import timedelta as td

    geom   = shapely_wkb.loads(wkb_bytes)
    minx, miny, maxx, maxy = geom.bounds
    buf    = SCALE_DEG
    region = ee.Geometry.BBox(minx - buf, miny - buf, maxx + buf, maxy + buf)

    storm_dates = []

    for year in SEARCH_YEARS:
        for month in range(1, 13):
            t_start = f"{year}-{month:02d}-01"
            # Last day of month
            if month == 12:
                t_end = f"{year + 1}-01-01"
            else:
                t_end = f"{year}-{month+1:02d}-01"

            try:
                ic = (
                    ee.ImageCollection('NASA/GPM_L3/IMERG_V07/FINAL/HALF_HOURLY')
                    .filterDate(t_start, t_end)
                    .filterBounds(region)
                    .select('precipitationCal')
                )
                n = ic.size().getInfo()
                if n == 0:
                    continue

                # Reduce to daily mean â€” find the day with max rain
                daily = ic.map(
                    lambda img: img.reduceRegion(
                        reducer=ee.Reducer.mean(),
                        geometry=region,
                        scale=int(SCALE_DEG * 111_000),
                        bestEffort=True
                    ).set('system:time_start', img.get('system:time_start'))
                )

                daily_info = daily.getInfo()
                if not daily_info:
                    continue

                # Find 30-min step with highest mean rainfall
                best_ts, best_val = None, -1
                for feat in daily_info:
                    val = feat.get('precipitationCal', 0) or 0
                    if val > best_val:
                        best_val = val
                        best_ts  = feat.get('system:time_start')

                if best_val < THRESH_1H or best_ts is None:
                    continue

                storm_dt   = datetime.utcfromtimestamp(best_ts / 1000)
                storm_date = storm_dt.date()

                # Exclude dates within BUFFER_DAYS of a known flood
                buffered = any(
                    abs((storm_date - fd).days) <= BUFFER_DAYS
                    for fd in flood_dates_set
                )
                if not buffered:
                    storm_dates.append(storm_dt)

            except Exception:
                continue

    return storm_dates

## 4. Generate no-flood events

For each sampled polygon:
1. Scan SEARCH_YEARS for candidate storm dates (GEE query)
2. For each storm date, download the IMERG matrix and compute intensities
3. Save the record with `is_flood = 0`

In [ ]:
# Identify already-processed polygons from saved batch files
done_files   = glob.glob(os.path.join(BATCH_DIR, 'no_flood_batch_*.parquet'))
done_uuids   = set()
if done_files:
    done_uuids = set(
        pd.concat([pd.read_parquet(f, columns=['source_uuid']) for f in done_files])
        ['source_uuid'].tolist()
    )
remaining_targets = targets[~targets['uuid'].isin(done_uuids)].copy()
print(f"Already processed: {len(done_uuids)} polygons")
print(f"Remaining targets: {len(remaining_targets)}")

t_session = time.time()
total_generated = 0

for batch_start in range(0, len(remaining_targets), BATCH_SIZE_POLY):
    batch       = remaining_targets.iloc[batch_start : batch_start + BATCH_SIZE_POLY]
    batch_label = f"{batch_start}_{batch_start + len(batch) - 1}"
    out_file    = os.path.join(BATCH_DIR, f'no_flood_batch_{batch_label}.parquet')

    if os.path.exists(out_file):
        continue

    records = []

    for _, row in batch.iterrows():
        # Collect all known flood dates for this polygon to define the exclusion buffer
        # Use uuid as the polygon identifier (geometry may differ slightly per record)
        flood_dates = set(
            df[df['uuid'] == row['uuid']]['start_date'].dt.date.tolist()
        )

        try:
            storm_dates = scan_for_storms(row['geometry'], flood_dates)
        except Exception as e:
            print(f"  Scan failed for uuid={row['uuid'][:8]}...: {e}")
            continue

        for storm_dt in storm_dates:
            try:
                matrix, mask, meta = extract_rain_matrix(
                    row['geometry'], storm_dt, product='final'
                )
                if matrix is None:
                    matrix, mask, meta = extract_rain_matrix(
                        row['geometry'], storm_dt, product='late'
                    )
                if matrix is None:
                    continue

                intensities = compute_intensities(matrix, mask)

                # Apply intensity threshold check
                if intensities.get('60_max_rainfall_intens', 0) < THRESH_1H:
                    continue
                if intensities.get('1440_max_rainfall_intens', 0) < THRESH_24H:
                    continue

                storm_date_str = storm_dt.strftime('%Y%m%d')
                record = {
                    'uuid'         : str(uuid.uuid4()),
                    'source_uuid'  : row['uuid'],
                    'event_id'     : f"{row['uuid']}_noflood_{storm_date_str}",
                    'start_date'   : storm_dt.strftime('%Y-%m-%d'),
                    'end_date'     : storm_dt.strftime('%Y-%m-%d'),
                    'geometry'     : row['geometry'],
                    'area_km2_original'  : row['area_km2_original'],
                    'area_km2_mollweide' : row['area_km2_mollweide'],
                    'urban_percentage'   : row['urban_percentage'],
                    'urban_built_up_area_m2': row.get('urban_built_up_area_m2', np.nan),
                    'polygon_total_area_m2' : row.get('polygon_total_area_m2', np.nan),
                    'upa_max'      : row.get('upa_max', np.nan),
                    'upa_p95'      : row.get('upa_p95', np.nan),
                    'upa_p99'      : row.get('upa_p99', np.nan),
                    'PFDI_p95'     : row.get('PFDI_p95', np.nan),
                    'PFDI_p99'     : row.get('PFDI_p99', np.nan),
                    'PFDI_max'     : row.get('PFDI_max', np.nan),
                    'imerg_matrix' : pickle.dumps(matrix),
                    'imerg_mask'   : pickle.dumps(mask),
                    'imerg_meta'   : pickle.dumps(meta),
                    'imerg_type'   : 'final',
                    'is_flood'     : 0,
                    **intensities
                }
                records.append(record)

            except Exception as e:
                print(f"  Matrix failed for storm {storm_dt.date()}: {e}")
                continue

    if records:
        pd.DataFrame(records).to_parquet(out_file, index=False)

    total_generated += len(records)
    pct = (batch_start + len(batch)) / len(remaining_targets) * 100
    print(f"Batch {batch_start//BATCH_SIZE_POLY + 1}  "
          f"| polygons {batch_start}â€“{batch_start+len(batch)-1}  "
          f"| generated {len(records)} events  "
          f"| total so far: {total_generated}  "
          f"| {pct:.1f}% done")

print(f"\nDone. Total no-flood events generated this session: {total_generated}")
print(f"Elapsed: {time.time()-t_session:.0f}s")

## 5. Summary of all generated no-flood events

In [ ]:
all_batch_files = glob.glob(os.path.join(BATCH_DIR, 'no_flood_batch_*.parquet'))
if all_batch_files:
    all_noflood = pd.concat(
        [pd.read_parquet(f, columns=[c for c in pd.read_parquet(f).columns
                                      if c not in ['imerg_matrix','imerg_mask','imerg_meta']])
         for f in all_batch_files],
        ignore_index=True
    )
    print(f"Total no-flood events: {len(all_noflood):,}")
    print(f"Unique source polygons: {all_noflood['source_uuid'].nunique()}")
    print(f"60-min intensity (median): {all_noflood['60_max_rainfall_intens'].median():.2f} mm/hr")
    print(all_noflood[['60_max_rainfall_intens', '1440_max_rainfall_intens',
                         'PFDI_max', 'urban_percentage']].describe())
else:
    print("No batch files found yet.")